**El Problema del viajero**

*Descripción general*

Dado un conjunto de **n ciudades** y las **distancias** entre cada par de ellas, el problema consiste en encontrar el **recorrido más corto** que:
- Parta de una ciudad de origen.
- Visite **exactamente una vez** cada ciudad.
- Regrese a la ciudad de origen al final del recorrido.

### Ejemplo visual

```
  A ---5--- B
  |       / |
  3     4   6
  |   /     |
  C ---2--- D

Ruta óptima: A → C → D → B → A  (costo = 3+2+6+5 = 16)
```

### Complejidad computacional
El TSP pertenece a la clase de problemas **NP-difícil**. Para `n` ciudades existen **(n-1)!/2** posibles rutas, lo que hace inviable la búsqueda exhaustiva para valores grandes de `n`:

| Ciudades (n) | Rutas posibles |
|:---:|---:|
| 5 | 12 |
| 10 | 181,440 |
| 20 | ~6 × 10¹⁶ |
| 50 | ~3 × 10⁶² |

*Objetivos, limitaciones y trabajos previos*

## Objetivos

El objetivo principal del TSP es:

**Minimizar** la función de costo (longitud total del recorrido):

$$\min \sum_{i=1}^{n} d(c_i, c_{i+1}) + d(c_n, c_1)$$

Donde:
- $c_i$ representa la ciudad en la posición $i$ del recorrido.
- $d(c_i, c_{i+1})$ es la distancia entre ciudades consecutivas.
- $d(c_n, c_1)$ es la distancia de retorno al origen.

### Objetivos secundarios
- Encontrar soluciones **aproximadas de alta calidad** en tiempo razonable.
- Diseñar algoritmos que **escalen** bien con el número de ciudades.
- Aplicar el modelo a problemas reales como logística, manufactura y redes.

## Limitaciones

### Limitaciones del problema
- **Explosión combinatoria:** El espacio de búsqueda crece factorialmente.
- **No existe algoritmo polinomial exacto** (a menos que P = NP).
- La versión **asimétrica** (donde $d(A,B) \neq d(B,A)$) es aún más difícil.

### Limitaciones del Algoritmo de Colonia de Hormigas (ACO)
- **Convergencia prematura:** Las hormigas pueden concentrarse en soluciones subóptimas.
- **Sensibilidad a parámetros:** Los resultados dependen fuertemente de $\alpha$, $\beta$ y la tasa de evaporación.
- **Tiempo de cómputo:** Requiere múltiples iteraciones para converger.
- **No garantiza optimalidad:** Es una metaheurística, no un método exacto.

## 4. Trabajos Previos y Métodos de Solución

### Métodos exactos
| Método | Descripción | Limitación |
|--------|-------------|------------|
| **Fuerza bruta** | Evalúa todas las rutas posibles | Solo viable para n ≤ 10 |
| **Programación dinámica** (Held-Karp, 1962) | Usa memoización, O(n²·2ⁿ) | Solo viable para n ≤ 20 |
| **Branch & Bound** | Poda el espacio de búsqueda | Aún exponencial en el peor caso |

### Heurísticas clásicas
- **Vecino más cercano:** Siempre ir a la ciudad no visitada más próxima. Simple pero subóptimo.
- **Inserción más barata:** Insertar ciudades minimizando el incremento de costo.
- **2-opt / 3-opt:** Mejorar una ruta intercambiando segmentos.

### Metaheurísticas (bio-inspiradas)
- **Algoritmos Genéticos (GA):** Holland, 1975. Evolución de soluciones mediante selección, cruce y mutación.
- **Recocido Simulado (SA):** Kirkpatrick, 1983. Acepta soluciones peores con cierta probabilidad.
- **Colonia de Hormigas (ACO):** Dorigo, 1992.
- **Optimización por Enjambre de Partículas (PSO):** Kennedy & Eberhart, 1995.

### El Algoritmo de Colonia de Hormigas (ACO)
Propuesto por **Marco Dorigo (1992)**, se inspira en el comportamiento de las hormigas reales al buscar comida. Las hormigas depositan **feromonas** en los caminos recorridos; los caminos más cortos acumulan más feromonas y atraen a más hormigas, generando un proceso de retroalimentación positiva que converge hacia buenas soluciones.

**Parámetros clave:**
- $\alpha$ (alfa): Influencia de las feromonas en la decisión.
- $\beta$ (beta): Influencia de la distancia (información heurística).
- $\rho$: Tasa de evaporación de feromonas (en este código: 0.1, ya que se multiplica por 0.9).

**Implementación del código**

In [4]:
import random, sys, math

# Nota: en lugar de matrices se usan listas de listas

# Genera una matriz de distancias de nCiudades x nCiudades
def matrizDistancias(nCiud, distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]

    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima * random.random()
            matriz[j][i] = matriz[i][j]

    return matriz

In [5]:
# Elige un paso de una hormiga, teniendo en cuenta las distancias
# y las feromonas y descartando las ciudades ya visitadas.
def eligeCiudad(dists, ferom, visitadas):
    # Se calcula la tabla de pesos de cada ciudad
    listaPesos  = []
    disponibles = []
    actual      = visitadas[-1]

    # Influencia de cada valor (alfa: feromonas; beta: distancias)
    alfa = 1.0
    beta = 0.5

    # El parámetro beta (peso de las distancias) es 0.5, alfa=1.0
    for i in range(len(dists)):
        if i not in visitadas:
            fer    = math.pow((1.0 + ferom[actual][i]), alfa)
            peso   = math.pow(1.0 / dists[actual][i], beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)

    # Se elige aleatoriamente una de las ciudades disponibles,
    # teniendo en cuenta su peso relativo.
    valor     = random.random() * sum(listaPesos)
    acumulado = 0.0
    i         = -1
    while valor > acumulado:
        i         += 1
        acumulado += listaPesos[i]

    return disponibles[i]

In [6]:
# Genera una "hormiga", que elegirá un camino teniendo en cuenta
# las distancias y los rastros de feromonas. Devuelve una tupla
# con el camino y su longitud.
def eligeCamino(distancias, feromonas):
    # La ciudad inicial siempre es la 0
    camino     = [0]
    longCamino = 0

    # Elegir cada paso según la distancia y las feromonas
    while len(camino) < len(distancias):
        ciudad      = eligeCiudad(distancias, feromonas, camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)

    # Para terminar hay que volver a la ciudad de origen (0)
    longCamino += distancias[camino[-1]][0]
    camino.append(0)

    return (camino, longCamino)

In [7]:
# Actualiza la matriz de feromonas siguiendo el camino recibido
def rastroFeromonas(feromonas, camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i+1]] += dosis

# Evapora todas las feromonas multiplicándolas por una constante
# = 0.9 (en otras palabras, el coeficiente de evaporación es 0.1)
def evaporaFeromonas(feromonas):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9

# Resuelve el problema del viajante de comercio mediante el
# algoritmo de la colonia de hormigas. Recibe una matriz de
# distancias y devuelve una tupla con el mejor camino que ha
# obtenido (lista de índices) y su longitud
def hormigas(distancias, iteraciones, distMedia):
    # Primero se crea una matriz de feromonas vacía
    n          = len(distancias)
    feromonas  = [[0 for i in range(n)] for j in range(n)]

    # El mejor camino y su longitud (inicialmente "infinita")
    mejorCamino     = []
    longMejorCamino = sys.maxsize

    # En cada iteración se genera una hormiga, que elige un camino,
    # y si es mejor que el mejor que teníamos, deja su rastro de
    # feromonas (mayor cuanto más corto sea el camino)
    for iter in range(iteraciones):
        (camino, longCamino) = eligeCamino(distancias, feromonas)

        if longCamino <= longMejorCamino:
            mejorCamino     = camino
            longMejorCamino = longCamino

        rastroFeromonas(feromonas, camino, distMedia / longCamino)

        # En cualquier caso, las feromonas se van evaporando
        evaporaFeromonas(feromonas)

    # Se devuelve el mejor camino que se haya encontrado
    return (mejorCamino, longMejorCamino)

In [8]:
# Generación de una matriz de prueba
numCiudades    = 10
distanciaMaxima = 10
ciudades        = matrizDistancias(numCiudades, distanciaMaxima)

# Obtención del mejor camino
iteraciones = 1000
distMedia   = numCiudades * distanciaMaxima / 2
(camino, longCamino) = hormigas(ciudades, iteraciones, distMedia)
print("Camino: ", camino)
print("Longitud del camino: ", longCamino)

Camino:  [0, 5, 9, 1, 6, 8, 3, 4, 7, 2, 0]
Longitud del camino:  23.09535865610549
